In [1]:
%load_ext autoreload
%autoreload 2

In [6]:
from src.openvino_encoder import OpenVINOEmbedder

model_dir = "models/rubert-tiny2_002"

embedder = OpenVINOEmbedder(model_dir, device="CPU")

texts = ["Пример текста для энкодинга", "Ещё один пример"]

In [ ]:
embeddings = embedder.encode(texts)
embeddings2 = await embedder.async_encode(texts)

print("Embeddings shape:", embeddings.shape)

Embeddings shape: (2, 312)


In [3]:
import openvino.properties as op

op.inference_num_threads()

'INFERENCE_NUM_THREADS'

In [51]:
import numpy as np

assert np.array_equal(embeddings, embeddings2)

In [6]:
import asyncio

inf_tok = embedder.tokenizer.create_infer_request()
inf_emb = embedder.model.create_infer_request()

In [ ]:
fut = asyncio.get_event_loop().create_future()


inf_tok.start_async(["пример запуска"])
await asyncio.to_thread(inf_tok.wait)
# inf_tok.wait()
toks = {n: v for n, v in zip(inf_tok.model_outputs)}

toks

[<ConstOutput: names[input_ids] shape[?,?] type: i64>,
 <ConstOutput: names[token_type_ids] shape[?,?] type: i64>,
 <ConstOutput: names[attention_mask] shape[?,?] type: i64>]

In [42]:
inf_tok.model_outputs[2].any_name

'attention_mask'

In [16]:
inf_tok.output_tensors

[<Tensor: shape[1,4] type: i64>,
 <Tensor: shape[1,4] type: i64>,
 <Tensor: shape[1,4] type: i64>]

In [43]:
inf_emb.start_async(inf_tok.output_tensors)

In [47]:
inf_emb.output_tensors[0].data

array([[[ 0.28840062, -0.2440517 , -0.55427194, ...,  0.58686113,
         -0.77706397, -0.33088195],
        [ 0.9238002 , -0.38073218, -0.8832226 , ...,  0.27579   ,
         -0.27247337, -0.6359046 ],
        [ 0.50280136, -0.07530709, -0.30521852, ...,  0.47144225,
         -0.39955178,  0.11307593],
        [-0.12758426, -1.4299176 , -0.7259816 , ...,  0.58542764,
         -0.1922034 , -0.45128915]]], shape=(1, 4, 312), dtype=float32)